# Choose initial values

Goose starts from the model's current state. Choose values with finite log
density on the correct support, then disperse the chains enough to reveal
different behavior.

In [1]:
import jax.numpy as jnp
import numpy as np
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl
import liesel.optim as opt

## Prepare the example

These examples require the regression `model`, including its transformation
and inference specifications, from {doc}`tutorials/md/01c-transform`.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


## Check starting values

For an existing Liesel `model`, call {meth}`~liesel.model.Model.diagnose`
to inspect its starting state.

Inspect values and log probabilities for NaN or infinity. For positive
parameters sampled with NUTS, use a suitable
{doc}`transformation <tutorials/md/01c-transform>` and initialize on its
sampling scale. Finite density is necessary but does not ensure finite gradients
or good mixing.

## Disperse the chains

For the regression `model` from {doc}`tutorials/md/01c-transform`:

In [3]:
joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
    jitter_method="additive",
)
model.vars["beta"].inference = joint
model.vars["log_sigma_sq"].inference = joint

In [4]:
results = gs.LieselMCMC(model).run_for_epochs(
    seed=2026,
    num_chains=4,
    adaptation=1000,
    posterior=1000,
    show_progress=False,
)

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


Each chain receives its own perturbation. The scalar jitter distribution draws
one perturbation per element of a vector parameter. A distribution with batch
or event dimensions must match the parameter's shape.

| `jitter_method` | Starting value | Check |
| --- | --- | --- |
| `"additive"` | Current value plus a draw. | The perturbation must respect support; a log parameter is often convenient. |
| `"multiplicative"` | Current value times a draw. | Use positive draws to keep a positive parameter positive. |
| `"replacement"` | The draw replaces the current value. | Choose a distribution with appropriate support and scale. |

`apply_jitter=True` applies these configured distributions. It does not
create random starts when `jitter_dist` is absent. Set `apply_jitter=False`
to use the existing starts without their configured perturbations.

## Start near a mode

For a model prepared for unconstrained optimization, fit first and then update
its state before building the sampler:

In [5]:
fit = opt.LieselOptim(
    model,
    optimizers="lbfgs",
    loss_monitor="train_full_data",
).fit()
model.state = model.update_state(fit.position_min_monitor, model.state)

liesel.optim.engine - INFO - Initializing optimization at epoch 0


Initializing:   0%|          | 0/1000 [00:00<?, ?it/s]

Train=1.448, Monitor=1.448:   1%|          | 10/1000 [00:01<02:51,  5.78it/s]

Train=1.448, Monitor=1.448:   2%|▏         | 19/1000 [00:02<01:46,  9.24it/s]

Fitting returns values separately; the assignment makes them the model's
starting state. The next `LieselMCMC(model).run_for_epochs(...)` call starts
there and applies any configured jitter. See {doc}`optimization` for fitting
choices. A mode can be a useful start, but it does not establish exploration
of the posterior or reveal other modes.

For individually supplied chain states, use
{meth}`~liesel.goose.EngineBuilder.set_initial_values` with
`multiple_chains=True`. Each state leaf then needs a leading chain axis.
Record starts and jitter settings along with the
{doc}`seed and environment <tutorials/md/05-reproducibility>`.